In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / "llm_OD_model" / "execution.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside LLM_OD_Llama3.1_package.")
MODEL_ROOT = PROJECT_ROOT / "llm_OD_model"
sys.path.insert(0, str(MODEL_ROOT))
sys.path.insert(0, str(MODEL_ROOT / "llm_network" / "viz"))
os.chdir(PROJECT_ROOT)


In [ ]:
import json
import pandas as pd

In [ ]:
exp = 'opinion_distr_theseus_same' # 'same'
llm = 'gemma34b' 
llm2 = 'mistral' if llm == 'llama3' else 'llama3'
min_size = 0.1
h = 1.0
rows = []
with open(f'results/theseus_{exp}_{llm}_0_PAH-min{min_size}-h{h}.jsonl') as f:
    init =  json.loads(f.readline())
    for i, l in enumerate(f):
        row = json.loads(l)
        rows.append(row)
        

In [ ]:
from llm_network.viz import OpinionArea
import os.path
from pathlib import Path

mode = ""

for llm in ['gemma3:b']:
    for min_size in [0.1, 0.3, 0.5]:
        for h in [0.0, 0.25, 0.5, 0.75, 1.0]:
            filename = f'results/{mode}theseus_{exp}_{llm}_0_PAH-min{min_size}-h{h}.jsonl'
            if os.path.exists(filename):
                img = OpinionArea(filename)
                output_file = Path(f"analysis/plots/opinion_distribution_trends/stacked/{mode}theseus_{exp}_{llm}_0_PAH-min{min_size}-h{h}.png")
                output_file.parent.mkdir(parents=True, exist_ok=True)
                img.plot(str(output_file), legend=False)
            else:
                print('not found', filename)




In [ ]:
from llm_network.viz import OpinionArea
import os.path
import matplotlib.pyplot as plt
from pathlib import Path
from llm_network.viz.plot_config import apply_plot_config
apply_plot_config()
from matplotlib.patches import Patch

llm = 'gemma34b'


# Define the labels and fixed colors (ideally match these with your original palette)
label_order = {
    0: "Strongly Disagree",
    1: "Disagree",
    2: "Mildly Disagree",
    3: "Neutral",
    4: "Mildly Agree",
    5: "Agree",
    6: "Strongly Agree",
}
# Choose the same color map and order used in the OpinionArea class
from matplotlib import cm
import numpy as np
colors = cm.get_cmap('RdYlGn')(np.linspace(0, 1, 7))

# Prepare legend handles
legend_handles = [Patch(facecolor=colors[i], edgecolor='black', label=label_order[i]) for i in range(7)]

Path("analysis/plots/opinion_distribution_trends/stacked").mkdir(parents=True, exist_ok=True)

for min_size in [0.1, 0.3, 0.5]:
    fig, axs = plt.subplots(1, 5, figsize=(25, 5), sharey=True, facecolor='white')
    h_values = [0.0, 0.25, 0.5, 0.75, 1.0]

    for i, h in enumerate(h_values):
        filename = f'results/{mode}theseus_{exp}_{llm}_0_PAH-min{min_size}-h{h}.jsonl'
        ax = axs[i]
        ax.set_facecolor('white')

        if os.path.exists(filename):
            print(f"Processing {filename}")
            img = OpinionArea(filename)
            img.plot(ax=ax, legend=False)
            ax.set_title(f"h = {h}", fontsize=14)
        else:
            print("Not found:", filename)
            ax.axis("off")
            ax.set_title(f"h = {h}\n(Missing)", fontsize=12)

    # Add a shared legend below all subplots
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=7,
        fontsize=20,
        frameon=False,
        bbox_to_anchor=(0.5, -0.05),
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.95])  # Adjust layout to leave space for the legend
    output_file = f"analysis/plots/opinion_distribution_trends/stacked/{mode}theseus_{exp}_{llm}_0_PAH-min{min_size}-grid.png"
    plt.savefig(output_file, bbox_inches='tight', facecolor='white')
    plt.close()


In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy import stats

from llm_network.viz.plot_config import apply_plot_config

results_root = Path("results")
output_dir = Path("analysis/plots/opinion_distribution_trends")
output_dir.mkdir(parents=True, exist_ok=True)

apply_plot_config()

folders = ["gemma_res_r1", "llama_res_r1"]
labels = {
    0: "Strongly Disagree",
    1: "Disagree",
    2: "Mildly Disagree",
    3: "Neutral",
    4: "Mildly Agree",
    5: "Agree",
    6: "Strongly Agree",
}
categories = list(labels.keys())
colors = plt.get_cmap("RdYlGn")(np.linspace(0, 1, len(categories)))
legend = [Line2D([0], [0], color=colors[i], lw=2, label=labels[i]) for i in categories]

files = []
for folder in folders:
    for path in sorted((results_root / folder).glob("*.jsonl")):
        match = re.match(r"theseus_(.+)_([0-9]+)_PAH-min([0-9.]+)-h([0-9.]+)\.jsonl", path.name)
        if match is None:
            continue
        exp, run, min_size, h = match.groups()
        files.append((folder, exp, int(run), float(min_size), float(h), path))

if len(files) == 0:
    raise FileNotFoundError("No result files found in results/gemma_res_r1 or results/llama_res_r1")

print("Found", len(files), "result files")

all_data = {}
for folder, exp, run, min_size, h, path in files:
    by_iteration = {}
    with open(path) as f:
        for line in f:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if "status" in row:
                by_iteration[row.get("iteration", -1)] = row["status"]

    series = []
    for iteration in sorted(by_iteration):
        values = [int(v) for v in by_iteration[iteration].values()]
        n = len(values)
        series.append([sum(v == c for v in values) / n for c in categories])

    all_data[(folder, exp, run, min_size, h)] = np.array(series).T

saved = []
for folder in folders:
    folder_data = [x for x in files if x[0] == folder]
    experiments = sorted({x[1] for x in folder_data})
    model_name = folder.replace("_res_r1", "")

    for exp in experiments:
        exp_data = [x for x in folder_data if x[1] == exp]
        runs = sorted({x[2] for x in exp_data})
        min_sizes = sorted({x[3] for x in exp_data})
        h_values = sorted({x[4] for x in exp_data})
        clean_exp = re.sub(r"[^A-Za-z0-9_.-]+", "_", exp)

        for min_size in min_sizes:
            fig, axs = plt.subplots(1, len(h_values), figsize=(4.5 * len(h_values), 4.2), sharey=True)
            axs = np.atleast_1d(axs)

            for j, h in enumerate(h_values):
                ax = axs[j]
                runs_data = []
                for run in runs:
                    key = (folder, exp, run, min_size, h)
                    if key in all_data:
                        runs_data.append(all_data[key])

                if len(runs_data) == 0:
                    ax.axis("off")
                    ax.set_title(f"h = {h} missing", fontsize=13)
                    continue

                min_len = min(data.shape[1] for data in runs_data)
                runs_data = np.array([data[:, :min_len] for data in runs_data]) * 100
                mean = runs_data.mean(axis=0)
                std = runs_data.std(axis=0, ddof=1) if len(runs_data) > 1 else np.zeros_like(mean)
                n_runs = len(runs_data)
                t_crit = stats.t.ppf(0.975, df=n_runs - 1) if n_runs > 1 else 0.0
                ci = t_crit * std / np.sqrt(n_runs)
                x = np.arange(min_len)

                for c in categories:
                    ax.fill_between(x, np.clip(mean[c] - ci[c], 0, 100), np.clip(mean[c] + ci[c], 0, 100), color=colors[c], alpha=0.12, linewidth=0)
                    ax.plot(x, mean[c], color=colors[c], linewidth=1.6)

                ax.set_title(f"h = {h} ({len(runs_data)} runs)", fontsize=13)
                ax.set_xlabel("Iterations")
                ax.set_xlim(0, max(0, min_len - 1))
                ax.set_ylim(0, 100)
                ax.set_yticks(np.arange(0, 101, 20))
                if j == 0:
                    ax.set_ylabel("Avg. % Agents (95% CI)")
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

            if min_size == max(min_sizes):
                fig.legend(
                    handles=legend,
                    loc="upper center",
                    bbox_to_anchor=(0.5, 0.98),
                    ncol=7,
                    frameon=False,
                    fontsize=11,
                    handlelength=1.2,
                    columnspacing=0.8,
                    borderaxespad=0.0,
                )
                plt.tight_layout(rect=[0, 0, 1, 0.88])
            else:
                plt.tight_layout()

            output_file = output_dir / f"{model_name}__{clean_exp}__PAH-min{min_size}-line-runs.png"
            plt.savefig(output_file, dpi=300, bbox_inches="tight", facecolor="white")
            plt.close(fig)

            if not output_file.exists():
                raise FileNotFoundError(f"Plot not saved: {output_file}")
            saved.append(output_file)
            print("saved", output_file)

print("Saved", len(saved), "line charts in", output_dir.resolve())
